# Cogni AP Tutor — production QLoRA
Select a GPU runtime, place the frozen `production_v1` dataset folder in Google Drive at `MyDrive/cogni/production_v1`, then run all cells. The test split is intentionally never loaded for training.

In [ ]:
!nvidia-smi
!pip -q install -U unsloth datasets trl peft accelerate bitsandbytes transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DATA_DIR = Path('/content/drive/MyDrive/cogni/production_v1')
OUTPUT_DIR = Path('/content/drive/MyDrive/cogni/models/ap_tutor_qwen3_1_7b_v1')
assert (DATA_DIR / 'train.jsonl').exists(), DATA_DIR
assert (DATA_DIR / 'validation.jsonl').exists(), DATA_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from datasets import load_dataset
files = {'train': str(DATA_DIR / 'train.jsonl'), 'validation': str(DATA_DIR / 'validation.jsonl')}
dataset = load_dataset('json', data_files=files)
print(dataset)
assert len(dataset['train']) >= 9000, 'Training split is unexpectedly small'
assert len(dataset['validation']) > 0

In [ ]:
from unsloth import FastLanguageModel
MODEL_ID = 'Qwen/Qwen3-1.7B'
MAX_SEQ_LENGTH = 2048
model, tokenizer = FastLanguageModel.from_pretrained(model_name=MODEL_ID, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=32, lora_alpha=32, lora_dropout=0, bias='none', target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'], use_gradient_checkpointing='unsloth', random_state=42)

In [ ]:
def format_chat(batch):
    return {'text': [tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False, enable_thinking=False) for messages in batch['messages']]}
train_ds = dataset['train'].map(format_chat, batched=True)
valid_ds = dataset['validation'].map(format_chat, batched=True)
print(train_ds[0]['text'][:1000])

In [ ]:
import torch
from transformers import TrainingArguments
from trl import SFTTrainer
args = TrainingArguments(output_dir=str(OUTPUT_DIR / 'checkpoints'), num_train_epochs=2, per_device_train_batch_size=2, per_device_eval_batch_size=2, gradient_accumulation_steps=16, learning_rate=2e-4, warmup_ratio=0.03, lr_scheduler_type='cosine', weight_decay=0.01, logging_steps=10, eval_strategy='steps', eval_steps=100, save_strategy='steps', save_steps=100, save_total_limit=3, load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False, fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(), optim='adamw_8bit', seed=42, report_to='none')
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_ds, eval_dataset=valid_ds, dataset_text_field='text', max_seq_length=MAX_SEQ_LENGTH, packing=False, args=args)
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')

In [ ]:
result = trainer.train()
print(result.metrics)
trainer.save_model(str(OUTPUT_DIR / 'adapter'))
tokenizer.save_pretrained(str(OUTPUT_DIR / 'adapter'))
model.save_pretrained_merged(str(OUTPUT_DIR / 'merged_16bit'), tokenizer, save_method='merged_16bit')
print('Saved to', OUTPUT_DIR)